In [1]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
nguyenngochonglinh_world_history_tiny_path = kagglehub.dataset_download('nguyenngochonglinh/world-history-tiny')

print('Data source import complete.')


Data source import complete.


# MegaRAG: Multimodal Knowledge Graph-based RAG on Kaggle

This notebook reproduces **MegaRAG** on Kaggle using the bundled repository [`reproduce_MegaRAG`](https://github.com/linhnguyen15492/reproduce_MegaRAG.git), which contains the complete source code for **MegaRAG**, **LightRAG**, and **MinerU**.

MegaRAG enables **global visual question answering** on documents by constructing a **Multimodal Knowledge Graph (MMKG)** combining graph-based reasoning with document page retrieval.

### Prerequisites
- **GPU Accelerator**: Tesla T4 / P100 or better (enable GPU in Kaggle settings)
- **Internet**: Turned ON in Kaggle settings
- **OpenAI API Key**: Added to Kaggle Secrets as `OPENAI_API_KEY` (or entered interactively)

**Paper**: [MegaRAG: Multimodal Graph-based Retrieval Augmented Generation (ACL 2026)](https://arxiv.org/abs/2512.20626)


## 1. System Setup & Clone Repository


In [1]:
import os
import sys
import shutil
import subprocess
from pathlib import Path
import torch

# Force PyTorch only & avoid CUDA memory fragmentation
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# Set base working directory (default Kaggle working directory is /kaggle/working)
if Path("/kaggle/working").exists():
    BASE_DIR = Path("/kaggle/working")
else:
    BASE_DIR = Path.cwd()

# Clone or locate the reproduce_MegaRAG repository
REPO_URL = "https://github.com/linhnguyen15492/reproduce_MegaRAG.git"
REPO_NAME = "reproduce_MegaRAG"
branch = "features/qwen_llm_with_vndoc"

if (BASE_DIR / "MegaRAG").exists() and (BASE_DIR / "MinerU").exists():
    REPO_DIR = BASE_DIR
elif (BASE_DIR / REPO_NAME / "MegaRAG").exists():
    REPO_DIR = BASE_DIR / REPO_NAME
    print(f"Updating repository {REPO_URL} inside {REPO_DIR}...")
    subprocess.run("git pull", shell=True, check=True, cwd=REPO_DIR)
    REPO_DIR = BASE_DIR / REPO_NAME
else:
    print(f"Cloning repository {REPO_URL} into {BASE_DIR}...")
    subprocess.run(f"git clone -b {branch} {REPO_URL}", shell=True, check=True, cwd=BASE_DIR)
    REPO_DIR = BASE_DIR / REPO_NAME

os.chdir(REPO_DIR)
print(f"✓ Working directory: {REPO_DIR}")

# Set component paths from the repository
mineru_dir = REPO_DIR / "MinerU"
megarag_dir = REPO_DIR / "MegaRAG"
lightrag_dir = REPO_DIR / "LightRAG"


PyTorch Version: 2.10.0+cu128
CUDA Available: True
GPU Device: Tesla T4
GPU Memory: 15.64 GB
Cloning repository https://github.com/linhnguyen15492/reproduce_MegaRAG.git into /kaggle/working...


Cloning into 'reproduce_MegaRAG'...


✓ Working directory: /kaggle/working/reproduce_MegaRAG


## 2. Install Required Dependencies & Local Packages


In [2]:
import subprocess
import sys

# 1. Install base dependencies for MinerU, LightRAG, and MegaRAG
required_packages = [
    "pyopenssl>=24.0.0",              # Fix Kaggle OpenSSL/cryptography mismatch
    "cryptography>=42.0.0",          # Fix GEN_EMAIL attribute error
    "transformers==4.51.3",
    "pillow>=10.2.0,<11.0.0",        # Avoid Pillow 11 typing issues with RapidTable
    "PyMuPDF==1.24.14",              # Required by MinerU/magic-pdf (<1.25.0)
    "pdfminer.six==20231228",        # Required by MinerU/magic-pdf
    "pypdfium2",                     # PDF rendering
    "rapid_table==1.0.3",            # Required for table extraction
    "loguru",                        # Logging
    "boto3",                         # MinerU dependency
    "timm",                          # Vision backbones
    "einops",                        # Tensor operations
    "openai>=1.50.0,<2.0.0",         # Modern OpenAI SDK (v1.x)
    "accelerate>=0.30.0,<2.0.0",     # PyTorch acceleration
    "beautifulsoup4>=4.12.0",        # HTML/XML parsing
    "opencv-python-headless",        # Headless OpenCV for server/Kaggle environments
    "ultralytics",                   # YOLO layout detection
    "doclayout-yolo",                # Document layout analysis
    "ftfy",                          # Text normalization
    "dill",                          # Serialization
    "shapely",                       # Bounding box geometry
    "pyclipper",                     # Polygon clipping
    "tiktoken",                      # Token counting for LLM
    "huggingface_hub",               # HuggingFace model download
    "matplotlib",                    # Visualization
    "rich",                          # Formatted console output
    "pyyaml",                        # YAML configuration parser
    "networkx",                      # Knowledge graph structures
]

print("Installing dependencies...")
try:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-warn-conflicts"] + required_packages)
    print("✓ Base dependencies installed successfully!")
except Exception as e:
    print(f"⚠️ Batch install note ({e}), installing individual packages...")
    for package in required_packages:
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
        except Exception:
            pass

# 2. Install bundled packages from repository in editable mode
print("Installing MinerU from repository...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", str(mineru_dir)])

print("Installing LightRAG from repository...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", str(lightrag_dir)])

print("Installing MegaRAG from repository...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", str(megarag_dir), "--no-deps"])

print("✓ All dependencies and packages installed successfully!")


Installing dependencies...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 103.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.8/19.8 MB 87.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 122.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 92.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 112.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 75.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 kB 49.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 MB 40.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
kaggle-environments 1.29.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
kaggle-environments 1.29.3 requires pydantic>=2.11.4, but you have pydantic 2.10.6 which is incompatible.
litellm 1.82.4 requires openai>=2.8.0, but you have openai 1.109.1 which is incompatible.
sigstore-models 0.0.6 requires pydantic>=2.12, but you have pydantic 2.10.6 which is incompatible.
a2a-sdk 0.3.26 requires pydantic>=2.11.3, but you have pydantic 2.10.6 which is incompatible.
mcp 1.27.0 requires pydantic<3.0.0,>=2.11.0, but you have pydantic 2.10.6 which is incompatible.
google-adk 1.29.0 requires pydantic<3.0.0,>=2.12.0, but you have pydantic 2.10.6 which is incompatible.


Installing LightRAG from repository...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 60.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.3/90.3 kB 7.0 MB/s eta 0:00:00
Installing MegaRAG from repository...
✓ All dependencies and packages installed successfully!


## 3. Setup MinerU Models & Configuration


In [4]:
import os
import shutil
import json as json_lib
import yaml
from pathlib import Path
import torch
from huggingface_hub import snapshot_download

print("Downloading MinerU model weights from HuggingFace...")

# 1. Download PDF-Extract-Kit models
pdf_extract_kit_path = snapshot_download(
    repo_id="opendatalab/PDF-Extract-Kit-1.0",
    allow_patterns=["models/*"],
)
models_dir = Path(pdf_extract_kit_path) / "models"
print(f"✓ PDF-Extract-Kit models: {models_dir}")

# 2. Download LayoutReader model
layoutreader_path = snapshot_download(
    repo_id="hantian/layoutreader",
)
layoutreader_model_dir = Path(layoutreader_path)
print(f"✓ LayoutReader model: {layoutreader_model_dir}")

# 3. Setup OCR model weights and configurations
ocr_models_dir = models_dir / "OCR" / "paddleocr_torch"
if ocr_models_dir.exists():
    v3_multi_det = ocr_models_dir / "Multilingual_PP-OCRv3_det_infer.pth"
    v4_ch_rec = ocr_models_dir / "ch_PP-OCRv4_rec_infer.pth"
    v4_server_rec = ocr_models_dir / "ch_PP-OCRv4_rec_server_infer.pth"

    # Setup Detection weights
    if v3_multi_det.exists():
        for det_target in ["ch_PP-OCRv3_det_infer.pth", "en_PP-OCRv3_det_infer.pth", "ch_PP-OCRv4_det_infer.pth"]:
            dest = ocr_models_dir / det_target
            shutil.copy2(v3_multi_det, dest)
            print(f"✓ Configured OCR detection model: {det_target}")

    # Setup Server Rec alias if needed
    if v4_ch_rec.exists() and not v4_server_rec.exists():
        shutil.copy2(v4_ch_rec, v4_server_rec)

    # Configure models_config.yml in MinerU so 'en' maps to matching 6625-vocab recognition model
    for cfg_yml in mineru_dir.glob("**/models_config.yml"):
        try:
            with open(cfg_yml, "r", encoding="utf-8") as f:
                y_data = yaml.safe_load(f)
            if "lang" in y_data and "en" in y_data["lang"]:
                y_data["lang"]["en"]["rec"] = "ch_PP-OCRv4_rec_infer.pth"
                y_data["lang"]["en"]["dict"] = "ppocr_keys_v1.txt"
            if "lang" in y_data and "latin" in y_data["lang"]:
                y_data["lang"]["latin"]["rec"] = "ch_PP-OCRv4_rec_infer.pth"
                y_data["lang"]["latin"]["dict"] = "ppocr_keys_v1.txt"
            with open(cfg_yml, "w", encoding="utf-8") as f:
                yaml.dump(y_data, f)
            print(f"✓ Configured {cfg_yml.name} with consistent OCR model shapes")
        except Exception as e:
            pass

# 4. Generate magic-pdf.json / mineru.json configuration
# Note: For non-formula documents (history, literature, general books), disabling formula recognition
# avoids UniMERNet MBart compatibility issues with modern transformers while running 3x faster!
device_mode = "cuda" if torch.cuda.is_available() else "cpu"
config_data = {
    "models-dir": str(models_dir),
    "device-mode": device_mode,
    "layoutreader-model-dir": str(layoutreader_model_dir),
    "layout-config": {
        "model": "doclayout_yolo"
    },
    "formula-config": {
        "enable": False
    },
    "table-config": {
        "model": "rapid_table",
        "sub_model": "slanet_plus",
        "enable": True,
        "max_time": 400
    },
    "latex-delimiter-config": {
        "display": {"left": "$$", "right": "$$"},
        "inline": {"left": "$", "right": "$"},
    },
}

config_paths = [
    Path.home() / "magic-pdf.json",
    Path("/root/magic-pdf.json"),
    Path.home() / "mineru.json",
    Path("/root/mineru.json"),
    mineru_dir / "magic-pdf.json",
    mineru_dir / "mineru.json",
    REPO_DIR / "magic-pdf.json",
]

for cfg_path in config_paths:
    try:
        cfg_path.parent.mkdir(parents=True, exist_ok=True)
        with open(cfg_path, "w", encoding="utf-8") as f:
            json_lib.dump(config_data, f, indent=4)
    except Exception:
        pass

print(f"✓ MinerU configuration generated successfully (device-mode: '{device_mode}')")


Fetching 185 files:   0%|          | 0/185 [00:00<?, ?it/s]

config.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/857 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/575 [00:00<?, ?B/s]

models/Layout/YOLO/doclayout_yolo_ft.pt:   0%|          | 0.00/40.7M [00:00<?, ?B/s]

...

.gitattributes: 0.00B [00:00, ?B/s]

README.md:   0%|          | 0.00/269 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/713M [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/713M [00:00<?, ?B/s]

✓ LayoutReader model: /root/.cache/huggingface/hub/models--hantian--layoutreader/snapshots/629be376d86fbab624ddc4020804a4e93b5515bc
✓ Configured OCR detection model: ch_PP-OCRv3_det_infer.pth
✓ Configured OCR detection model: en_PP-OCRv3_det_infer.pth
✓ Configured OCR detection model: ch_PP-OCRv4_det_infer.pth
✓ Configured models_config.yml with consistent OCR model shapes
✓ MinerU configuration generated successfully (device-mode: 'cuda')


## 4. Setup API Keys & Environment Variables


In [5]:
import os
import sys
import shutil
import re
import getpass
from pathlib import Path

# 1. Retrieve OpenAI API Key from Kaggle Secrets, Environment, or User Input
openai_api_key = os.environ.get("OPENAI_API_KEY")

if not openai_api_key:
    try:
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        openai_api_key = user_secrets.get_secret("OPENAI_API_KEY")
        print("✓ OpenAI API Key loaded from Kaggle Secrets.")
    except Exception:
        openai_api_key = None

if not openai_api_key:
    openai_api_key = getpass.getpass("Enter your OpenAI API Key: ")

os.environ["OPENAI_API_KEY"] = openai_api_key

# 2. Update PATH with Python and MinerU bin locations
py_bin_dir = str(Path(sys.executable).parent)
extra_paths = [
    py_bin_dir,
    "/root/.local/bin",
    str(Path.home() / ".local" / "bin"),
    "/usr/local/bin",
    "/opt/conda/bin",
]
for p in extra_paths:
    if p not in os.environ.get("PATH", ""):
        os.environ["PATH"] = f"{p}:{os.environ.get('PATH', '')}"

# 3. Write env.sh in MegaRAG directory
magic_pdf_bin = shutil.which("magic-pdf") or shutil.which("mineru") or py_bin_dir
magic_pdf_bin_dir = str(Path(magic_pdf_bin).parent) if magic_pdf_bin else str(Path(sys.executable).parent)

env_sh_content = f"""#!/usr/bin/env bash
export OPENAI_API_KEY="{openai_api_key}"
export MINERU_PATH="{magic_pdf_bin_dir}"
"""

env_file = megarag_dir / "env.sh"
with open(env_file, "w", encoding="utf-8") as f:
    f.write(env_sh_content)
print(f"✓ Configured {env_file}")


✓ OpenAI API Key loaded from Kaggle Secrets.
✓ Configured /kaggle/working/reproduce_MegaRAG/MegaRAG/env.sh


## 5. Prepare Example Data (`world_history_tiny`)

Load `World_History_Volume_1.pdf` and `queries.txt` from `/kaggle/input/datasets` (or `/kaggle/input/`) into the workspace `data/` directory (alongside `LightRAG`, `MegaRAG`, `MinerU`).


In [6]:
import os
import shutil
from pathlib import Path

# Setup data directory directly under REPO_DIR (alongside LightRAG, MegaRAG, MinerU)
data_dir = REPO_DIR / "data"
data_dir.mkdir(parents=True, exist_ok=True)
pdf_dir = Path("/root/.cache/kagglehub/datasets/nguyenngochonglinh/world-history-tiny")

pdf_file = data_dir / "World_History_Volume_1.pdf"
queries_file = data_dir / "queries_short.txt"

# 1. Search for dataset files in /kaggle/input/datasets, /kaggle/input, or existing repo paths
kaggle_input = Path("/kaggle/input")
search_roots = [
    kaggle_input / "datasets",
    kaggle_input,
    megarag_dir / "egs" / "world_history_tiny" / "data",
    pdf_dir
]

pdf_source = None
queries_source = None

for root in search_roots:
    if root.exists():
        # Look for PDF file
        if not pdf_source:
            pdf_matches = list(root.glob("**/World_History_Volume_1.pdf"))
            if pdf_matches:
                pdf_source = pdf_matches[0]
        # Look for queries file
        if not queries_source:
            query_matches = list(root.glob("**/queries_short.txt"))
            if query_matches:
                queries_source = query_matches[0]

# Copy PDF to data directory
if pdf_source and pdf_source.exists():
    shutil.copy2(pdf_source, pdf_file)
    print(f"✓ Found and copied PDF from: {pdf_source}")
elif pdf_file.exists():
    print(f"✓ PDF already present in data directory: {pdf_file}")
else:
    print(f"⚠️ PDF file 'World_History_Volume_1.pdf' not found in /kaggle/input/datasets!")
    print(f"   Please make sure the dataset is added to the Kaggle notebook input.")

# Copy or generate queries.txt
if queries_source and queries_source.exists():
    shutil.copy2(queries_source, queries_file)
    print(f"✓ Found and copied queries from: {queries_source}")
elif not queries_file.exists():
    sample_queries = """- Question 1: What is the Byzantine Empire?
- Question 2: When did the Roman Empire fall?
- Question 3: Who was Alexander the Great?
- Question 4: What was the Silk Road?
- Question 5: Describe ancient Egyptian civilization.
"""
    with open(queries_file, "w", encoding="utf-8") as f:
        f.write(sample_queries.strip() + "\n")
    print(f"✓ Created benchmark queries file at: {queries_file}")
else:
    print(f"✓ Queries file ready: {queries_file}")

if pdf_file.exists():
    print(f"✓ Input PDF ready: {pdf_file} ({pdf_file.stat().st_size / (1024 * 1024):.2f} MB)")


✓ Found and copied PDF from: /kaggle/input/datasets/nguyenngochonglinh/world-history-tiny/World_History_Volume_1.pdf
✓ Found and copied queries from: /kaggle/input/datasets/nguyenngochonglinh/world-history-tiny/queries_short.txt
✓ Input PDF ready: /kaggle/working/reproduce_MegaRAG/data/World_History_Volume_1.pdf (190.04 MB)


## 6. Build Multimodal Knowledge Graph (MMKG)

Outputs and intermediate assets are stored inside the run directory `<pdf_name>_run/` located directly under `REPO_DIR` (alongside `LightRAG`, `MegaRAG`, `MinerU`), containing `dumps/` and `exp/`.

We split the MMKG construction into 4 distinct steps:
- **6.1**: Parse PDF using MinerU (`magic-pdf`) $\rightarrow$ `<pdf_name>_run/dumps/`
- **6.2**: Convert PDF pages to images (`pdf2img.py`)
- **6.3**: Build Page Assets manifest (`build_page_assets.py`)
- **6.4**: Construct MMKG (`construct_mmkg.py`) $\rightarrow$ `<pdf_name>_run/exp/`


### 6.1. Parse PDF with MinerU

Extract text, layout, tables, and embedded images from the PDF into `<pdf_name>_run/dumps/`.


In [7]:
import os
import sys
import shutil
import subprocess
from pathlib import Path

pdf_path = data_dir / "World_History_Volume_1.pdf"
pdf_name = pdf_path.stem

# Setup dedicated run directory at the base repo level (alongside LightRAG, MegaRAG, MinerU)
run_dir = REPO_DIR / f"{pdf_name}_run"
dumps_dir = run_dir / "dumps"
dumps_dir.mkdir(parents=True, exist_ok=True)
exp_dir = run_dir / "exp" / pdf_name
exp_dir.mkdir(parents=True, exist_ok=True)
config_file = megarag_dir / "egs" / "world_history_tiny" / "conf" / "addon_params.yaml"

os.chdir(run_dir)
print(f"Run directory (root level): {run_dir}")
print(f"Dumps directory: {dumps_dir}")

# Parse PDF using MinerU magic-pdf CLI (pages 0-9 for quickstart)
print("\n--- [Step 6.1] Parsing PDF with MinerU ---")
magic_pdf_bin = shutil.which("magic-pdf") or f"{sys.executable} -m magic_pdf.tools.cli"
cmd_parse = f"{magic_pdf_bin} -p {pdf_path} -o {dumps_dir} -m auto -e 9"
print(f"Running: {cmd_parse}\n")

# Run directly so all logs, progress bars, and root error messages are shown in full
subprocess.run(cmd_parse, shell=True, check=True)

# Standardize output folders (ensure dumps/pdf_name/auto contains all parsed assets)
doc_dir = dumps_dir / pdf_name
auto_dir = doc_dir / "auto"
auto_dir.mkdir(parents=True, exist_ok=True)

content_lists = list(dumps_dir.rglob("*_content_list.json"))
if content_lists:
    src_dir = content_lists[0].parent
    if src_dir.resolve() != auto_dir.resolve():
        for item in src_dir.iterdir():
            dest = auto_dir / item.name
            if not dest.exists():
                shutil.copytree(item, dest) if item.is_dir() else shutil.copy2(item, dest)

print(f"\n✓ Step 6.1 completed: PDF parsing done!")
print(f"Verified assets under '{auto_dir}':")
for file in sorted(auto_dir.iterdir()):
    print(f"  - {file.name}{'/' if file.is_dir() else f' ({file.stat().st_size / 1024:.1f} KB)'}")


Run directory (root level): /kaggle/working/reproduce_MegaRAG/World_History_Volume_1_run
Dumps directory: /kaggle/working/reproduce_MegaRAG/World_History_Volume_1_run/dumps

--- [Step 6.1] Parsing PDF with MinerU ---
Running: /usr/local/bin/magic-pdf -p /kaggle/working/reproduce_MegaRAG/data/World_History_Volume_1.pdf -o /kaggle/working/reproduce_MegaRAG/World_History_Volume_1_run/dumps -m auto -e 9

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.


2026-08-22 10:01:19.125 | INFO     | magic_pdf.data.dataset:__init__:157 - lang: None
2026-08-22 10:01:20.028 | INFO     | magic_pdf.libs.pdf_check:detect_invalid_chars:67 - cid_count: 0, text_len: 9101, cid_chars_radio: 0.0
2026-08-22 10:01:21.151 | INFO     | magic_pdf.model.doc_analyze_by_custom_model:may_batch_image_analyze:275 - gpu_memory: 15 GB, batch_ratio: 8
2026-08-22 10:01:21.154 | INFO     | magic_pdf.model.pdf_extract_kit:__init__:68 - DocAnalysis init, this may take some times, layout_model: doclayout_yolo, apply_formula: False, apply_ocr: True, apply_table: True, table_model: rapid_table, lang: None
2026-08-22 10:01:21.154 | INFO     | magic_pdf.model.pdf_extract_kit:__init__:82 - using device: cuda
2026-08-22 10:01:21.155 | INFO     | magic_pdf.model.pdf_extract_kit:__init__:86 - using models_dir: /root/.cache/huggingface/hub/models--opendatalab--PDF-Extract-Kit-1.0/snapshots/ed6b654c018d742e65a17671e379c5e6ecc87ec9/models
2026-08-22 10:01:23.172 | INFO     | magic_pdf.


✓ Step 6.1 completed: PDF parsing done!
Verified assets under '/kaggle/working/reproduce_MegaRAG/World_History_Volume_1_run/dumps/World_History_Volume_1/auto':
  - World_History_Volume_1.md (10.0 KB)
  - World_History_Volume_1_content_list.json (16.0 KB)
  - World_History_Volume_1_layout.pdf (2491.6 KB)
  - World_History_Volume_1_middle.json (1287.2 KB)
  - World_History_Volume_1_model.json (460.8 KB)
  - World_History_Volume_1_origin.pdf (2468.8 KB)
  - World_History_Volume_1_spans.pdf (2590.1 KB)
  - images/


### 6.2. Convert PDF Pages to Images

Convert the PDF pages into JPEG format using `pdf2img.py`.


In [8]:
import sys
import subprocess
from pathlib import Path
import shutil

print("--- [Step 6.2] Converting PDF pages to images ---")
pdf2img_py = megarag_dir / "egs" / "utils" / "pdf2img.py"
doc_dir = dumps_dir / pdf_name

target_img_dirs = [
    doc_dir / "auto" / "page_images",
    doc_dir / "page_images",
]

primary_img_dir = target_img_dirs[0]
primary_img_dir.mkdir(parents=True, exist_ok=True)

cmd_img = f"{sys.executable} {pdf2img_py} {pdf_path} {primary_img_dir} --dpi 150 --jpeg --end-page 10 --jobs 4"
print(f"Running: {cmd_img}")
subprocess.run(cmd_img, shell=True, check=True)

# Synchronize page_images to doc_dir / page_images
for t_dir in target_img_dirs[1:]:
    t_dir.mkdir(parents=True, exist_ok=True)
    for img in primary_img_dir.glob("*.jpg"):
        dest_img = t_dir / img.name
        if not dest_img.exists():
            shutil.copy2(img, dest_img)

print(f"✓ Step 6.2 completed: Saved {len(list(primary_img_dir.glob('*.jpg')))} page images to {primary_img_dir}")


--- [Step 6.2] Converting PDF pages to images ---
Running: /usr/bin/python3 /kaggle/working/reproduce_MegaRAG/MegaRAG/egs/utils/pdf2img.py /kaggle/working/reproduce_MegaRAG/data/World_History_Volume_1.pdf /kaggle/working/reproduce_MegaRAG/World_History_Volume_1_run/dumps/World_History_Volume_1/auto/page_images --dpi 150 --jpeg --end-page 10 --jobs 4
Rendering pages 1–10 of 788 at 150 DPI to JPEG using 4 processes…
Done ✔
✓ Step 6.2 completed: Saved 0 page images to /kaggle/working/reproduce_MegaRAG/World_History_Volume_1_run/dumps/World_History_Volume_1/auto/page_images


### 6.3. Build Page Assets Manifest

Merge the parsed text, tables, figure images, and page images into a unified `pages_content.json` manifest.


In [9]:
import sys
import subprocess
from pathlib import Path
import shutil

print("--- [Step 6.3] Building Page Assets Manifest ---")
build_assets_py = megarag_dir / "egs" / "utils" / "build_page_assets.py"
doc_dir = dumps_dir / pdf_name

# 1. Discover working directory containing parsed content
candidate_dirs = [
    doc_dir / "auto",
    doc_dir / "ocr",
    doc_dir / "txt",
    doc_dir,
]
found_lists = list(dumps_dir.rglob("*_content_list.json"))
for cl in found_lists:
    if cl.parent not in candidate_dirs:
        candidate_dirs.insert(0, cl.parent)

working_dir = None
for c_dir in candidate_dirs:
    if c_dir.exists() and list(c_dir.glob("*_content_list.json")):
        working_dir = c_dir
        break

if not working_dir:
    working_dir = doc_dir / "auto"

# 2. Ensure page_images exists inside working_dir
if not (working_dir / "page_images").exists():
    for pimg_dir in [doc_dir / "page_images", doc_dir / "auto" / "page_images"]:
        if pimg_dir.exists() and pimg_dir != (working_dir / "page_images"):
            shutil.copytree(pimg_dir, working_dir / "page_images")
            break

page_manifest = doc_dir / "pages_content.json"

cmd_assets = f"{sys.executable} {build_assets_py} --working-dir {working_dir} --output {page_manifest}"
print(f"Running: {cmd_assets}")
subprocess.run(cmd_assets, shell=True, check=True)
print(f"✓ Step 6.3 completed: Page assets manifest generated at {page_manifest} ({page_manifest.stat().st_size / 1024:.1f} KB)")


--- [Step 6.3] Building Page Assets Manifest ---
Running: /usr/bin/python3 /kaggle/working/reproduce_MegaRAG/MegaRAG/egs/utils/build_page_assets.py --working-dir /kaggle/working/reproduce_MegaRAG/World_History_Volume_1_run/dumps/World_History_Volume_1/auto --output /kaggle/working/reproduce_MegaRAG/World_History_Volume_1_run/dumps/World_History_Volume_1/pages_content.json
Total embedded images collected: 2
Saved page-asset manifest → /kaggle/working/reproduce_MegaRAG/World_History_Volume_1_run/dumps/World_History_Volume_1/pages_content.json
✓ Step 6.3 completed: Page assets manifest generated at /kaggle/working/reproduce_MegaRAG/World_History_Volume_1_run/dumps/World_History_Volume_1/pages_content.json (12.3 KB)


### 6.4. Construct Multimodal Knowledge Graph

Extract multimodal entities, relationships, and embeddings to build the Knowledge Graph using `construct_mmkg.py` into `<pdf_name>_run/exp/`.


In [10]:
import sys
import time
import subprocess
from pathlib import Path

print("--- [Step 6.4] Constructing Multimodal Knowledge Graph ---")
construct_mmkg_py = megarag_dir / "egs" / "utils" / "construct_mmkg.py"

cmd_mmkg = f"{sys.executable} {construct_mmkg_py} --config-file {config_file} --working-dir {exp_dir} --input-dir {page_manifest}"

print(f"Running: {cmd_mmkg}")

t0 = time.time()

try:
    # stdout/stderr được stream trực tiếp ra notebook
    result = subprocess.run(
        cmd_mmkg,
        shell=True,
        check=False,
        text=True,
    )

    elapsed = time.time() - t0

    print("\n" + "=" * 100)
    print("MMKG CONSTRUCTION SUMMARY")
    print("=" * 100)
    print(f"Return code : {result.returncode}")
    print(f"Elapsed     : {elapsed:.1f}s")

    if result.returncode != 0:
        raise RuntimeError(
            f"construct_mmkg.py failed with return code {result.returncode}"
        )

    print("\n✓ Step 6.4 completed successfully!")
    print(f"✓ MMKG Construction finished in {elapsed:.1f} seconds!")

except KeyboardInterrupt:
    elapsed = time.time() - t0
    print("\n\n⚠ MMKG construction interrupted by user.")
    print(f"Elapsed: {elapsed:.1f}s")
    raise


--- [Step 6.4] Constructing Multimodal Knowledge Graph ---
Running: /usr/bin/python3 /kaggle/working/reproduce_MegaRAG/MegaRAG/egs/utils/construct_mmkg.py --config-file /kaggle/working/reproduce_MegaRAG/MegaRAG/egs/world_history_tiny/conf/addon_params.yaml --working-dir /kaggle/working/reproduce_MegaRAG/World_History_Volume_1_run/exp/World_History_Volume_1 --input-dir /kaggle/working/reproduce_MegaRAG/World_History_Volume_1_run/dumps/World_History_Volume_1/pages_content.json


A new version of the following files was downloaded from https://huggingface.co/Alibaba-NLP/gme-Qwen2-VL-2B-Instruct:
- modeling_gme_qwen2vl.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
Fetching 3 files: 100%|██████████| 3/3 [00:39<00:00, 13.15s/it]
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
Loading checkpoint shards: 100%|██████████| 3/3 [00:04<00:00,  1.43s/it]


⠴ 🔄 Updating package: nano-vectordb

Rerank is enabled but no rerank_model_func provided. Reranking will be skipped.


25h


Final Token Usage: LLM call count: 20, Prompt tokens: 134577, Completion tokens: 9178, Total tokens: 143755.

MMKG CONSTRUCTION SUMMARY
Return code : 0
Elapsed     : 281.4s

✓ Step 6.4 completed successfully!
✓ MMKG Construction finished in 281.4 seconds!


## 7. Query with MegaRAG

Following Step 4 of MegaRAG README (`run_quering.sh`): Query the Multimodal Knowledge Graph using `query_mmkg.py`.


In [11]:
import os
import sys
import time
import subprocess
from pathlib import Path

query_script = megarag_dir / "egs" / "utils" / "query_mmkg.py"
results_dir = exp_dir / "results"
results_dir.mkdir(parents=True, exist_ok=True)
results_file = results_dir / "results.json"

cmd_query = f"""{sys.executable} {query_script} \
    --config-file {config_file} \
    --working-dir {exp_dir} \
    --input-queries {queries_file} \
    --output-file {results_file} \
    --concurrency 1 \
    --max-retries 3 \
    --query-delay 5.0 \
    --resume
"""

print(f"Running queries:\n{cmd_query}\n")
t0 = time.time()
subprocess.run(cmd_query, shell=True, check=True)
print(f"\n✓ Querying completed in {time.time() - t0:.1f} seconds!")


Running queries:
/usr/bin/python3 /kaggle/working/reproduce_MegaRAG/MegaRAG/egs/utils/query_mmkg.py     --config-file /kaggle/working/reproduce_MegaRAG/MegaRAG/egs/world_history_tiny/conf/addon_params.yaml     --working-dir /kaggle/working/reproduce_MegaRAG/World_History_Volume_1_run/exp/World_History_Volume_1     --input-queries /kaggle/working/reproduce_MegaRAG/data/queries_short.txt     --output-file /kaggle/working/reproduce_MegaRAG/World_History_Volume_1_run/exp/World_History_Volume_1/results/results.json     --concurrency 1     --max-retries 3     --query-delay 5.0     --resume




Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
Loading checkpoint shards: 100%|██████████| 3/3 [00:04<00:00,  1.47s/it]
Rerank is enabled but no rerank_model_func provided. Reranking will be skipped.


[0] (22.29s) Integrating primary sources into lessons about Early Human Evolution and Migration can significantly enhance critical thinking among students by allowing them to engage with historical evidence directly. Here are some strategies to incorporate primary sources effectively in such lessons, fostering a deeper understanding of the material:

### 1. **Analysis of Archaeological Evidence**
Encourage students to examine archaeological findings related to early human evolution, such as tools, fossils, and cave art. By analyzing these artifacts, students can infer patterns of behavior, social organization, and technological advancements of early humans. Utilize maps and images that represent these archaeological sites, directing students to consider how geographic factors influenced early human migration.

### 2. **Use of Genetic Studies**
Incorporate genetic data as primary sources, such as ancestral DNA analysis which provides insights into migration patterns. Students can evalua

## 8. View Results & Knowledge Graph Analysis


In [12]:
import json
from pathlib import Path
import networkx as nx

# 1. Display Query Results
if results_file.exists():
    with open(results_file, "r", encoding="utf-8") as f:
        results = json.load(f)

    print("=" * 80)
    print("MEGARAG QUERY RESULTS")
    print("=" * 80)

    if isinstance(results, list):
        for i, res in enumerate(results, 1):
            print(f"\n{'─' * 80}")
            print(f"Query {i}: {res.get('query', 'N/A')}")
            print(f"{'─' * 80}")
            if "answer" in res:
                print(f"Answer:\n{res['answer']}\n")
            if "retrieved_context" in res:
                print(f"Retrieved Context:\n{res['retrieved_context'][:300]}...\n")
    elif isinstance(results, dict):
        for query, ans in results.items():
            print(f"\n{'─' * 80}")
            print(f"Query: {query}")
            print(f"{'─' * 80}")
            answer_text = ans.get("answer", ans) if isinstance(ans, dict) else ans
            print(f"Answer:\n{answer_text}\n")
    print("=" * 80)

# 2. Knowledge Graph Analysis
graphml_files = list(exp_dir.glob("**/*.graphml"))
if graphml_files:
    G = nx.read_graphml(graphml_files[0])
    print(f"\nKnowledge Graph Overview ({graphml_files[0].name}):")
    print(f"  • Total Entities (Nodes): {G.number_of_nodes()}")
    print(f"  • Total Relationships (Edges): {G.number_of_edges()}")

    degrees = dict(G.degree())
    top_nodes = sorted(degrees.items(), key=lambda x: x[1], reverse=True)[:5]
    print("\nTop 5 Most Connected Entities:")
    for rank, (node, deg) in enumerate(top_nodes, 1):
        entity_type = G.nodes[node].get("entity_type", "entity")
        print(f"  {rank}. [{entity_type}] {node} ({deg} connections)")


MEGARAG QUERY RESULTS

────────────────────────────────────────────────────────────────────────────────
Query: results
────────────────────────────────────────────────────────────────────────────────
Answer:
[{'index': 0, 'question': 'How can primary sources be integrated into lessons about Early Human Evolution and Migration to foster critical thinking?', 'answer': "Integrating primary sources into lessons about Early Human Evolution and Migration can significantly enhance critical thinking among students by allowing them to engage with historical evidence directly. Here are some strategies to incorporate primary sources effectively in such lessons, fostering a deeper understanding of the material:\n\n### 1. **Analysis of Archaeological Evidence**\nEncourage students to examine archaeological findings related to early human evolution, such as tools, fossils, and cave art. By analyzing these artifacts, students can infer patterns of behavior, social organization, and technological adva

In [13]:
import os
import shutil
from IPython.display import FileLink, display

def download_kaggle_working(output_filename="kaggle_output.zip", root_dir="/kaggle/working/reproduce_MegaRAG/World_History_Volume_1_run"):
    """
    Nén và tạo link tải toàn bộ thư mục Kaggle working.
    """
    # Xoá đuôi .zip nếu người dùng vô tình nhập vào tên file
    base_name = output_filename.rsplit('.zip', 1)[0]
    zip_path = f"/kaggle/working/{base_name}.zip"
    
    # 1. Xoá file zip cũ nếu đã tồn tại để tránh nén lặp vô tận
    if os.path.exists(zip_path):
        os.remove(zip_path)
    
    print("⏳ Đang nén dữ liệu từ /kaggle/working...")
    
    # 2. Tạo file zip tạm ở /kaggle/tmp để tránh nén chính file zip đang tạo
    os.makedirs("/kaggle/temp_archive", exist_ok=True)
    temp_zip_base = f"/kaggle/temp_archive/{base_name}"
    
    shutil.make_archive(
        base_name=temp_zip_base,
        format='zip',
        root_dir=root_dir
    )
    
    # 3. Di chuyển file zip về /kaggle/working để FileLink có thể truy cập
    shutil.move(f"{temp_zip_base}.zip", zip_path)
    shutil.rmtree("/kaggle/temp_archive")
    
    # 4. Hiển thị link tải về
    file_size_mb = os.path.getsize(zip_path) / (1024 * 1024)
    print(f"✅ Nén hoàn tất! Dung lượng: {file_size_mb:.2f} MB")
    print("👉 Nhấp vào đường link bên dưới để tải về:")
    
    display(FileLink(f"{base_name}.zip"))

In [14]:
download_kaggle_working("results.zip")

⏳ Đang nén dữ liệu từ /kaggle/working...
✅ Nén hoàn tất! Dung lượng: 10.29 MB
👉 Nhấp vào đường link bên dưới để tải về:


/kaggle/working/reproduce_MegaRAG/World_History_Volume_1_run/results.zip